# 05 — The same pipeline in PySpark (DE)

Original plan was Colab; **this machine turned out to have Java 25 (Temurin)**, so the
notebook runs LOCALLY with `PYSPARK_PYTHON` pointed at the repo venv (the Windows
'python3' worker fix). Everything else per spec: scale-up replicate trick, Spark twin
of M6's rolling form, honest timing table, batch-vs-streaming.

*Note:* `winutils.exe` warnings are expected on Windows for local mode without HADOOP_HOME;
they don't affect in-memory CSV workloads.

In [1]:
import os
import sys
from pathlib import Path

VENV_PY = str(Path.cwd() / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_PYTHON"] = VENV_PY
os.environ["PYSPARK_DRIVER_PYTHON"] = VENV_PY
print(sys.executable)

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Scripts\python.exe


In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("cs2-m9")
    .config("spark.driver.host", "localhost")
    .config("spark.sql.shuffle.partitions", "8")  # else 200 partitions skew the timing
    .config("spark.pyspark.python", os.environ["PYSPARK_PYTHON"])
    .config("spark.executorEnv.PYSPARK_PYTHON", os.environ["PYSPARK_PYTHON"])
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(spark.version)

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


## 1-2. Load + scale-up (replicate trick)

In [3]:
REPO = Path.cwd()
series = pd.read_csv(REPO / "outputs" / "series_clean.csv")
series["datetime"] = pd.to_datetime(series["datetime"], utc=True, format="ISO8601")

# long format (2 rows per match) = the pandas ground truth for form
t1 = series[["match_id", "datetime", "team1", "winner"]].rename(columns={"team1": "team"})
t1["won"] = (t1["winner"] == t1["team"]).astype(float)
t2 = series[["match_id", "datetime", "team2", "winner"]].rename(columns={"team2": "team"})
t2["won"] = (t2["winner"] == t2["team"]).astype(float)
long_pd = (
    pd.concat([t1, t2], ignore_index=True)
    .sort_values(["team", "datetime", "match_id"], kind="mergesort")
    .reset_index(drop=True)
)

N_REP = 200
rep = pd.concat([long_pd.assign(replicate_id=i) for i in range(N_REP)], ignore_index=True)
print(f"original long rows: {len(long_pd):,} | replicated: {len(rep):,}")

original long rows: 19,840 | replicated: 3,968,000


## 3. Spark rolling form (rowsBetween(-5, -1)) + correctness proof

In [4]:
# NOTE: the Spark twin of pandas `rolling(5).mean().shift(1)` is rowsBetween(-5, -1):
# after the shift, the pandas window covers up to 5 PRE-match rows, not 4.
def spark_form5(sdf):
    w = Window.partitionBy("team").orderBy("datetime", "match_id").rowsBetween(-5, -1)
    return sdf.withColumn("form5", F.avg("won").over(w))


# correctness proof on the ORIGINAL rows: pandas (shifted rolling mean) vs Spark
def pandas_form5(long):
    g = long.groupby("team")["won"].transform(lambda s: s.rolling(5, min_periods=1).mean().shift(1))
    return g


pdf_small = long_pd.copy()
pdf_small["form5_pd"] = pandas_form5(pdf_small)
sdf = spark.createDataFrame(pdf_small[["match_id", "team", "datetime", "won"]])
out = spark_form5(sdf).select("match_id", "team", "form5").toPandas()
out = out.rename(columns={"form5": "form5_sp"})
merged = pdf_small.merge(out, on=["match_id", "team"])
merged = merged.dropna(subset=["form5_sp"])  # Spark rowsBetween -> null on each team's first row
diff = (merged["form5_pd"] - merged["form5_sp"]).abs()
print(f"compared {len(merged)} rows | max |diff| = {diff.max():.2e}")
assert diff.max() < 1e-9, "Spark rolling form disagrees with pandas!"
print("CORRECTNESS PROOF: Spark window form == pandas rolling form")

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


compared 19048 rows | max |diff| = 0.00e+00
CORRECTNESS PROOF: Spark window form == pandas rolling form


## 4. Timing: pandas vs Spark at 3 scales (honest numbers)

In [5]:
def pandas_job(df):
    t0 = time.perf_counter()
    _ = df.groupby("team")["won"].transform(lambda s: s.rolling(5, min_periods=1).mean().shift(1))
    return time.perf_counter() - t0


def spark_job(df):
    t0 = time.perf_counter()
    sdf = spark.createDataFrame(df[["match_id", "team", "datetime", "won"]])
    sdf = sdf.withColumn(
        "form5",
        F.avg("won").over(
            Window.partitionBy("team").orderBy("datetime", "match_id").rowsBetween(-5, -1)
        ),
    )
    _ = sdf.select("match_id", "team", "form5").toPandas()
    return time.perf_counter() - t0


rows = []
rng = np.random.default_rng(0)
for scale_rows in (len(long_pd), 100_000, 500_000):
    take = rep.iloc[:scale_rows] if scale_rows <= len(rep) else None
    if take is None:
        mult = int(np.ceil(scale_rows / len(long_pd)))
        take = pd.concat(
            [long_pd.assign(replicate_id=i) for i in range(mult)], ignore_index=True
        ).iloc[:scale_rows]
    tp = time.perf_counter()
    pandas_job(take)
    p_sec = time.perf_counter() - tp
    ts = time.perf_counter()
    spark_job(take)
    s_sec = time.perf_counter() - ts
    # full float precision so the contract speedup == pandas_sec / spark_sec holds exactly
    rows.append(
        {
            "scale_rows": scale_rows,
            "pandas_sec": p_sec,
            "spark_sec": s_sec,
            "speedup": p_sec / s_sec,
        }
    )
timing = pd.DataFrame(rows)
timing.to_csv(REPO / "outputs" / "m9_spark_vs_pandas.csv", index=False)
timing

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\c

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\c

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\c

C:\Users\illya\OneDrive\Desktop\cs2-match-analytics\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,scale_rows,pandas_sec,spark_sec,speedup
0,19840,0.099172,0.234645,0.422647
1,100000,0.098057,4.996950,0.019623
2,500000,0.119597,17.986113,0.006649


**The honest read (spec §1.4):** at ~20k rows pandas wins — the JVM startup, the
python-worker handshake, and the toPandas() materialization are fixed overheads Spark
pays on every job. Spark's window engine wins only when data outgrows one process's
memory. At 10-500k rows on a laptop, pandas is simply the right tool.

## 5. Batch vs streaming (this project)

**Batch (what we have):** nightly results dump -> parquet -> walk-forward rating update.
Everything retrains/re-rates from a frozen snapshot; failures just re-run yesterday's job.
What breaks: late-arriving results (score corrections after publication) silently change
ratings retroactively unless you version the snapshots.

**Streaming (in-play odds would need it):** round-by-round events -> Kafka -> stateful
rating update per round. What breaks: exactly-once semantics under rebalancing (a rating
double-updated on replay = wrong live odds) and out-of-order events needing watermarking.
The rating update is trivially a fold; the hard part is the *delivery guarantees*, not
the math.

In [6]:
spark.stop()
print("spark session stopped")

spark session stopped
